# Lab 4 — Quantize + Fine-Tune (QLoRA)

**Day 1 Afternoon | ~90 minutes | Colab T4 GPU**

---

## ⚠️ Switch to a T4 before you run anything

```
Runtime → Change runtime type → T4 GPU → Save
```

Do it now. If you load weights on CPU first and switch later, you will lose the runtime and start over.

---

In Lab 3 you worked out what a model costs to hold in memory. You computed `params × 2` for BF16 and `params × 0.5` for INT4, and you estimated how many users would fit on a 16 GB GPU. All of it was arithmetic on a CPU.

Today those numbers get tested on real hardware, and then we go further: you will take a model that has been squeezed into 4 bits and **train** it anyway.

That combination is the point of this lab. Quantization makes a model small enough to load. LoRA makes it cheap enough to fine-tune. Together — QLoRA — they put a job that used to need a datacenter onto a free Colab GPU, and the thing you ship at the end is a file you could email.

## What you will walk out with

1. What "4-bit" actually means, down to where the sixteen values come from and why they are not evenly spaced.
2. Why quantization is measured in memory saved and paid for in speed, with both numbers in front of you.
3. Why a rank-16 adapter can teach a 1.5-billion-parameter model new behaviour while touching 2% of a layer.
4. Why the adapter starts as a mathematical no-op, and why that matters.
5. What you actually deploy: a small adapter, a merged directory, or a GGUF file, and when each one is right.

## The route

```
Part A              Part B              Part C              Part D            Bonus
──────────────      ──────────────      ──────────────      ──────────────    ──────────
What 4 bits    →    Measure it     →    LoRA and       →    Ship the     →    Pruning,
really means        on the T4           QLoRA               artifact          and why it
(CPU, on real       (VRAM, speed,       (the math, then     (adapter,         loses
 Qwen weights)       quality)            the training)       merge, GGUF)

~20 min             ~20 min             ~25 min             ~20 min           ~8 min
```

Parts A and C start on the **CPU** with small tensors you can actually read, then move to the GPU once the idea is clear. If your Colab GPU gets reclaimed mid-lab, the concept cells still run.

**Coming from Lab 3:** same model family, one size up — `Qwen2.5-1.5B-Instruct` instead of 0.5B. The `q_proj` weight tensor you measured in Lab 3 comes back in Part A as the thing we quantize by hand.

In [ ]:
import sys
%pip install -q uv
!uv pip install -q --python {sys.executable} "transformers>=5" torch accelerate bitsandbytes peft "trl>=0.16" datasets
!uv pip install -q --python {sys.executable} --upgrade torchao

`torchao` is a PyTorch quantization helper that some Colab images ship in an old version, and the stale copy clashes with current bitsandbytes. The extra upgrade line is there to head that off.

Now confirm you are actually on a GPU. If this prints a warning, stop — everything after Part A needs real VRAM.

In [ ]:
# Cell 0 — Are we on a GPU?
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name}  ({total:.1f} GB)")
else:
    print("No GPU found.")
    print("Runtime -> Change runtime type -> T4 GPU -> Save, then re-run from the top.")

---

# Part A — What 4 bits actually means

**~20 minutes · runs on CPU**

Everyone says "we quantized it to 4 bits" as though the sentence explains itself. It does not. Four bits gives you **sixteen possible values**, total. A weight that used to be one of billions of representable float16 numbers now has to be one of sixteen.

So there are three questions worth answering before you type `load_in_4bit=True`:

1. Sixteen values covering what range?
2. Placed where?
3. Shared across how many weights?

Those three questions are exactly the arguments of `BitsAndBytesConfig`. Let's answer them on a real tensor before we use the library.

We will borrow the model you took apart in Lab 3.

In [ ]:
# Cell A1 — Load the Lab 3 model on CPU and grab one weight matrix
from transformers import AutoModelForCausalLM

small = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", dtype=torch.float32)
W = small.model.layers[0].self_attn.q_proj.weight.detach()

print(f"Tensor : layers[0].self_attn.q_proj.weight")
print(f"Shape  : {tuple(W.shape)}   ({W.numel():,} weights)")

Same `q_proj` you measured in Lab 3, when you found it was 896 wide and its `k_proj` neighbour was only 128. Now we care about the *values* inside it rather than its shape.

In [ ]:
# Cell A2 — What do trained weights actually look like?
print(f"mean : {W.mean():+.5f}")
print(f"std  : {W.std():.5f}")
print(f"min  : {W.min():+.5f}")
print(f"max  : {W.max():+.5f}")
print()
sigma = W.std()
for k in (1, 2, 3):
    inside = (W.abs() < k * sigma).float().mean()
    print(f"within {k} sigma of zero : {inside:.1%} of all weights")
print()
print(f"but the largest weight is {W.abs().max() / sigma:.0f} sigma out")

Two facts, and everything about quantization design follows from them.

**Trained weights cluster hard around zero.** Mean is essentially zero, and 82% of these weights sit within one standard deviation of it. Plot them and you get a bell curve. Nothing about this layer is special; weight decay and normalized initialization produce the same shape across essentially every trained network.

**And yet the extremes are far out.** The standard deviation is about 0.067, but the biggest weight is around 1.2. That is roughly **eighteen standard deviations**. A handful of weights live enormously far from where almost all of their neighbours are.

Hold on to the tension there. Almost everything is tiny; a few things are huge. Now try to cover that with sixteen values.

In [ ]:
# Cell A3 — Attempt one: sixteen evenly spaced values across the whole tensor
def quantize_dequantize(x, levels, block_size=None):
    """Round each weight to the nearest available level, then convert back."""
    flat = x.flatten()
    if block_size is None:
        block_size = flat.numel()                      # one shared scale for everything
    pad = (-flat.numel()) % block_size
    flat = torch.cat([flat, torch.zeros(pad)])
    blocks = flat.view(-1, block_size)

    scale = blocks.abs().max(dim=1, keepdim=True).values.clamp(min=1e-12)
    normalized = blocks / scale                        # now in [-1, 1]
    nearest = (normalized.unsqueeze(-1) - levels).abs().argmin(dim=-1)
    return (levels[nearest] * scale).flatten()[:x.numel()].view_as(x)


uniform16 = torch.linspace(-1, 1, 16)                  # sixteen evenly spaced values
print("The sixteen levels:", [f"{v:+.3f}" for v in uniform16[:4]], "...", [f"{v:+.3f}" for v in uniform16[-2:]])

W_whole = quantize_dequantize(W, uniform16)
print(f"\nmean absolute error: {(W - W_whole).abs().mean():.6f}")

Now think about what that scale had to do. It was set by the single largest weight in the tensor, around 1.2, so the sixteen levels got stretched across ±1.2. The gap between neighbouring levels is therefore about 0.16.

But 82% of the weights are smaller than 0.067. They are **all** closer to zero than one step of the grid. Almost the entire tensor collapses onto the same two or three levels, and the fine structure that the training process spent GPU-months learning is gone.

One outlier wrecked the resolution for everyone else. The fix is to stop letting it.

In [ ]:
# Cell A4 — Attempt two: a separate scale for every 64 weights
W_blocks = quantize_dequantize(W, uniform16, block_size=64)

err_whole = (W - W_whole).abs().mean()
err_block = (W - W_blocks).abs().mean()

print(f"{'one scale for the whole tensor':<32s}: {err_whole:.6f}")
print(f"{'one scale per 64 weights':<32s}: {err_block:.6f}")
print(f"\n{err_whole / err_block:.1f}x more accurate, same 4 bits per weight")

That is **block-wise quantization**, and it is the single biggest idea in making 4-bit work. An outlier now only ruins the 63 weights it shares a block with, instead of the entire matrix. Every other block gets a scale matched to its own contents.

It is not free. Each block needs its scale stored alongside it:

In [ ]:
# Cell A5 — What block-wise storage actually costs
for block_size in (64, 128, 256):
    bits_per_weight = 4 + 16 / block_size        # 4-bit weight + one fp16 scale per block
    print(f"block of {block_size:>3}: {bits_per_weight:.3f} bits per weight")

print()
print("A 1.5B model at 4.25 bits/weight:", f"{1.5e9 * 4.25 / 8 / 1e9:.2f} GB")
print("A 1.5B model at exactly 4 bits  :", f"{1.5e9 * 4 / 8 / 1e9:.2f} GB")

Four and a quarter bits, not four. That extra quarter-bit is pure overhead — scales, not weights — and on a 7B model it is hundreds of megabytes.

Which is what `bnb_4bit_use_double_quant=True` is for. It quantizes the scales themselves, storing them in 8 bits with their own second-level scale, and claws back roughly 0.4 bits per weight. You will set that flag in Part B. Now you know what it is buying.

That answers questions 1 and 3: sixteen values, scaled per 64-weight block. One question left, and it is the one hiding inside the name **NormalFloat4**.

## Why the levels are not evenly spaced

Go back to the shape of the data. Weights are bunched around zero and thin out toward the edges. Evenly spaced levels ignore that completely — they spend just as much resolution out at ±0.9, where almost nothing lives, as they do at ±0.02, where most of the tensor is.

So don't space them evenly. Place them at the **quantiles of a normal distribution**: tightly packed near zero where the weights are, spread out near the edges where they are not. Every level then does roughly equal work.

That is NF4, from the QLoRA paper (Dettmers et al., 2023). The sixteen values below are the ones the paper derived, and they are what bitsandbytes uses when you write `bnb_4bit_quant_type='nf4'`.

In [ ]:
# Cell A6 — The actual NF4 levels, and what they cost in error
nf4 = torch.tensor([-1.0, -0.6962, -0.5251, -0.3949, -0.2844, -0.1848, -0.0911, 0.0,
                    0.0796, 0.1609, 0.2461, 0.3379, 0.4407, 0.5626, 0.7230, 1.0])

err_uniform = (W - quantize_dequantize(W, uniform16, block_size=64)).abs().mean()
err_nf4     = (W - quantize_dequantize(W, nf4,       block_size=64)).abs().mean()

print(f"uniform INT4, blocks of 64 : {err_uniform:.6f}")
print(f"NF4,          blocks of 64 : {err_nf4:.6f}")
print(f"\n{(1 - err_nf4 / err_uniform):.0%} less error, for exactly the same 4 bits")

In [ ]:
# Cell A7 — Where NF4 spends its resolution
print("gap between neighbouring levels:\n")
print(f"{'':12s}{'near zero':>12s}{'at the edge':>14s}")
print(f"{'NF4':12s}{nf4[8] - nf4[7]:>12.4f}{nf4[15] - nf4[14]:>14.4f}")
print(f"{'uniform':12s}{uniform16[8] - uniform16[7]:>12.4f}{uniform16[15] - uniform16[14]:>14.4f}")

There it is, in two numbers. NF4 resolves differences as small as 0.08 near zero, where four out of five weights live, and tolerates gaps of 0.28 out at the edges where hardly any do. Uniform spacing offers a flat 0.13 everywhere, which is too coarse where it matters and wasted where it does not.

Twenty percent less error, no extra storage, purely from putting the sixteen values in better places.

## You can now read the config

Everything in Part B's quantization config maps to something you just measured:

| Argument | What it means, now that you have seen it |
|---|---|
| `load_in_4bit=True` | Sixteen levels per weight instead of float16's range |
| `bnb_4bit_quant_type='nf4'` | Use the quantile-spaced levels from cell A6, not the even ones |
| `bnb_4bit_use_double_quant=True` | Quantize the block scales too, recovering that extra quarter-bit from A5 |
| `bnb_4bit_compute_dtype=torch.bfloat16` | Store in 4 bits, but **compute** in bf16 |

That last one is doing more than it looks, and it is the reason for a result that surprises people in Part B. Keep it in mind.

#### ✅ Checkpoint A

- [ ] How many distinct values can a single 4-bit weight take?
- [ ] Why does one scale for a whole tensor lose so much more than one scale per 64 weights?
- [ ] Why are the NF4 levels bunched up near zero?
- [ ] Why is a "4-bit" model actually more than 4 bits per weight, and what reduces that?

<details>
<summary>Answers</summary>

**Sixteen.** 2⁴.

**Because the scale is set by the largest weight in its group.** Across a whole tensor that means one 18-sigma outlier stretches the grid for millions of ordinary weights, and they all collapse onto the same few levels. Per 64 weights, an outlier only damages its own block.

**Because that is where the weights are.** 82% of them sit within one sigma of zero. Levels placed at normal quantiles put resolution where the data is instead of spreading it evenly across mostly-empty space.

**Because every block stores a scale alongside it** — 4 + 16/64 = 4.25 bits per weight. Double quantization compresses those scales and gets back about 0.4 of it.

</details>

---

# Part B — Measure it on the T4

**~20 minutes · needs the GPU**

Theory is done. Now load the same model twice, in FP16 and in NF4, and measure three things: **memory, speed, and quality.**

Two of those will go the way you expect. One will not, and it is the most useful result in this part.

A note on measuring VRAM honestly. `torch.cuda.memory_allocated(0)` reports what is allocated *at this instant*, which is easy to read at the wrong moment and easy to pollute with leftovers from a previous cell. We use `reset_peak_memory_stats()` before each load and `max_memory_allocated()` after, so what you get is the true high-water mark for that model.

In [ ]:
# Cell B1 — Free the CPU model from Part A, we are done with it
import gc

del small, W
gc.collect()

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"   # change to 0.5B if you hit a VRAM error
print(f"Benchmark model: {MODEL_ID}")

In [ ]:
# Cell B2 — Load the FP16 baseline and record its peak VRAM
import time
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(0)

start = time.time()
model_fp16 = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")
load_fp16 = time.time() - start
vram_fp16 = torch.cuda.max_memory_allocated(0) / 1e9

print(f"loaded in {load_fp16:.1f}s")
print(f"peak VRAM: {vram_fp16:.2f} GB")

Check that number against the prediction you made in Lab 3. A 1.5B model at 2 bytes per parameter should be about 3.1 GB, and the weights are the bulk of what you just saw.

Next, a helper to measure generation speed. It runs the same prompt a few times and averages, because a single timing on a shared Colab GPU is noise.

In [ ]:
# Cell B3 — A benchmark helper (defines only, no output)
def benchmark(model, prompt, n_runs=3, max_new_tokens=80):
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    times = []
    for _ in range(n_runs):
        start = time.time()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=False, pad_token_id=tokenizer.eos_token_id)
        times.append(time.time() - start)

    average = sum(times) / len(times)
    new_tokens = out.shape[1] - prompt_len
    return {"speed": new_tokens / average,
            "text": tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)}


print("benchmark() defined.")

In [ ]:
# Cell B4 — Time the FP16 model
PROMPT = "Explain quantization vs pruning in 3 bullet points."

stats_fp16 = benchmark(model_fp16, PROMPT)
print(f"FP16: {stats_fp16['speed']:.1f} tokens/sec")
print(f"\n{stats_fp16['text'][:300]}")

Now free it. A T4 has about 15 GB and we are about to load a second copy of the same model, so this cell is not optional housekeeping — skip it and the NF4 load will fail.

Deleting the Python reference is not enough on its own. PyTorch holds freed blocks in its own caching allocator, so `empty_cache()` is what actually returns them.

In [ ]:
# Cell B5 — Release the FP16 model
del model_fp16
gc.collect()
torch.cuda.empty_cache()

print(f"still allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

In [ ]:
# Cell B6 — Load the same model in NF4
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                       # 16 levels per weight   (cell A3)
    bnb_4bit_quant_type="nf4",               # quantile-spaced levels (cell A6)
    bnb_4bit_use_double_quant=True,          # compress the scales    (cell A5)
    bnb_4bit_compute_dtype=torch.bfloat16,   # store 4-bit, compute bf16
)

torch.cuda.reset_peak_memory_stats(0)

start = time.time()
model_nf4 = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
load_nf4 = time.time() - start
vram_nf4 = torch.cuda.max_memory_allocated(0) / 1e9

print(f"loaded in {load_nf4:.1f}s")
print(f"peak VRAM: {vram_nf4:.2f} GB")

In [ ]:
# Cell B7 — Time the NF4 model and compare everything
stats_nf4 = benchmark(model_nf4, PROMPT)

print(f"{'':16s}{'FP16':>12s}{'NF4':>12s}")
print("-" * 40)
print(f"{'VRAM (GB)':16s}{vram_fp16:>12.2f}{vram_nf4:>12.2f}")
print(f"{'tokens/sec':16s}{stats_fp16['speed']:>12.1f}{stats_nf4['speed']:>12.1f}")
print(f"{'load time (s)':16s}{load_fp16:>12.1f}{load_nf4:>12.1f}")
print()
print(f"NF4 uses {vram_fp16 / vram_nf4:.1f}x less memory "
      f"and runs at {stats_nf4['speed'] / stats_fp16['speed']:.2f}x the speed")

## The result nobody warns you about

Memory went down by roughly 3×. Good, expected, that is the whole point.

**Speed probably went down too.** If your NF4 row shows fewer tokens per second than FP16, nothing is broken and you did not misconfigure anything. That is the normal outcome on a T4, and here is why.

Look again at `bnb_4bit_compute_dtype=torch.bfloat16`. The weights are *stored* in 4 bits, but no GPU has a matmul instruction that takes 4-bit inputs and does something useful with them here. So on every single forward pass, for every layer, bitsandbytes **dequantizes the weights back to bf16**, does an ordinary matmul, and throws the dequantized copy away.

You are paying for that unpacking, constantly. What you bought with it is that the 4-bit version is the one sitting in VRAM between operations.

So state the trade honestly:

- **Quantization is a memory technique.** It is not a speed technique.
- It makes a model **fit** — on a smaller GPU, or alongside more concurrent users, or with room left over for the KV cache you measured in Lab 3.
- If the model already fits comfortably in FP16, quantizing it may well make things worse.
- Speedups from 4-bit do exist, but they come from kernels built for it (Marlin, AWQ, GPTQ with fused ops) rather than from the bit width by itself.

There is a second thing worth noticing: NF4 often **loads faster** than FP16, because loading is dominated by moving bytes and there are four times fewer of them.

And quality? Read the two samples below. On a task this forgiving you will struggle to tell them apart, which is the finding — 4× smaller, and you cannot see the difference by eye.

In [ ]:
# Cell B8 — Same prompt, both precisions
print("FP16:")
print(" ", stats_fp16["text"][:400])
print("\nNF4:")
print(" ", stats_nf4["text"][:400])

#### ✅ Checkpoint B

- [ ] How much VRAM did each version need, and what ratio is that?
- [ ] Which was faster, and can you explain why without using the word "should"?
- [ ] What is quantization actually buying you?
- [ ] Could you tell the two outputs apart?

<details>
<summary>Answers</summary>

**Roughly 3× less for NF4.** Not the full 4× — activations, the KV cache, and CUDA's own workspace are unquantized and identical in both runs.

**FP16 is usually faster**, because NF4 unpacks every weight to bf16 on every forward pass. That dequantization is real work that FP16 never does.

**Fit, not speed.** Room for a bigger model, more users, or a longer context on hardware you already have.

**Almost certainly not.** Which is the reason NF4 is worth using at all.

</details>

---

# Part C — LoRA, and why QLoRA works

**~25 minutes · starts on CPU, finishes on the GPU**

You now have a 1.5B model taking up about a gigabyte of VRAM. Time to teach it something.

The obvious approach is full fine-tuning: unfreeze all 1.5 billion parameters and train. Price that out before dismissing it. You need the weights, plus a gradient for every weight, plus optimizer state — Adam keeps two running averages per parameter. In FP16 with Adam that is roughly **16 bytes per parameter**, so a 1.5B model needs around 24 GB before you have loaded a single training example. Your T4 has 15.

There is also something quietly wasteful about it. You are going to show this model ten examples. Does adjusting all 1.5 billion parameters really sound like the right-sized tool?

## The LoRA bet

LoRA (Hu et al., Microsoft, 2021) starts from an observation about what fine-tuning does. Take the weight matrix before fine-tuning and after, and subtract. The difference, ΔW, turns out to have **low intrinsic rank** — it can be closely approximated by the product of two much smaller matrices.

If that is true, you never have to represent ΔW in full. Freeze W, and learn the two small matrices instead:

```
output = x·W  +  x·A·B
         ─────    ───────
         frozen   trained
```

`A` takes you from the layer's width down to a small rank `r`, and `B` brings you back up. Everything gradients flow through is inside those two.

Let's size it for a real layer in the model you have loaded.

In [ ]:
# Cell C1 — LoRA shapes for one attention projection
d = 1536          # Qwen2.5-1.5B hidden size
r = 16            # the rank we will use

W_full = torch.zeros(d, d)        # frozen base weight, stand-in
A = torch.randn(r, d) * 0.01      # down to rank r
B = torch.zeros(d, r)             # back up to full width

for label, tensor, status in [("base W", W_full, "frozen"),
                              ("LoRA A", A, "trained"),
                              ("LoRA B", B, "trained")]:
    print(f"{label}  {str(tuple(tensor.shape)):>14s}  {tensor.numel():>10,} params   {status}")

trainable = A.numel() + B.numel()
print(f"\ntrainable: {trainable:,} of {W_full.numel():,} = {trainable / W_full.numel():.2%} of this layer")

Two percent. The arithmetic is worth seeing plainly: a full matrix costs `d × d`, while the pair costs `2 × d × r`. With `d` at 1536 and `r` at 16, that is 2.36 million against 49 thousand.

Notice `B` is initialized to **zeros**. That is not an oversight.

In [ ]:
# Cell C2 — Why B starts at zero
x = torch.randn(4, d)

base_output = x @ W_full.T
lora_contribution = (x @ A.T) @ B.T

print(f"largest value the adapter contributes at init: {lora_contribution.abs().max():.1f}")
print(f"output unchanged by the adapter? {torch.equal(base_output, base_output + lora_contribution)}")

`True`. Anything times zero is zero, so at the moment you attach a LoRA adapter the model's behaviour is **bit-for-bit identical** to the base model.

That is a deliberate and rather elegant property. Training starts from exactly the model you already trusted, and every change from that point is something the optimizer chose. Compare with initializing both matrices randomly: the model would start by producing garbage and spend the early steps clawing its way back to where it began.

So what does `r` actually cost you?

In [ ]:
# Cell C3 — Picking the rank
print(f"{'rank':>6s}{'params/layer':>16s}{'% of layer':>14s}")
for rank in (4, 8, 16, 32, 64):
    n = 2 * d * rank
    print(f"{rank:>6d}{n:>16,}{n / (d * d):>13.2%}")

Rank is the capacity dial. Small `r` buys you fewer parameters, a smaller adapter file and faster training, at the cost of how much change it can absorb. Large `r` is the reverse, and past a point you are paying for capacity the task will never use.

Sensible defaults: **8 or 16 for style, tone, and format changes**, which is most of what people actually fine-tune for. Push to 32 or 64 when teaching genuinely new behaviour. We use 16.

And `lora_alpha`? The adapter's output is scaled by `alpha / r` before being added. With `alpha=32` and `r=16` that is 2. The convention of setting alpha to twice the rank exists so that changing `r` does not silently change the effective learning rate of the adapter.

## Why this composes with quantization

Here is the part that makes QLoRA more than the sum of its pieces.

The base model is frozen. Frozen means no gradients, and no gradients means **the precision of those weights barely matters** — they are only ever read, in a forward pass, and Part B showed they get dequantized to bf16 for the matmul anyway.

So put the frozen 1.5 billion parameters in 4 bits, and keep the 49-thousand-parameter adapters, which *do* need gradients and optimizer state, in bf16. Every byte of precision goes where the training actually happens.

```
frozen base, 4-bit   ~1 GB     read-only, quality barely affected
LoRA adapters, bf16  ~35 MB    gradients + Adam state live here
```

That is QLoRA. Full fine-tuning needed roughly 24 GB. This fits in about 2, which is the only reason a free Colab T4 can do it at all.

Time to attach it.

In [ ]:
# Cell C4 — Prepare the quantized model for training
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_nf4.config.use_cache = False
model_nf4 = prepare_model_for_kbit_training(model_nf4)

print("ready for training")

Two lines, both worth a sentence.

**`use_cache = False`** should make you suspicious, given that Lab 3 spent an entire part proving the KV cache is the difference between 2 seconds and 43. There is no contradiction here, just a different job. The KV cache exists for *autoregressive generation*, where you produce one token at a time and want to avoid recomputing the past. Training does a single forward pass over a complete sequence that is already known, so there is nothing to reuse. Keeping the cache on here would just consume VRAM and fight with gradient checkpointing. Turn it back on for inference.

**`prepare_model_for_kbit_training`** does the small unglamorous fixes that make gradients behave on a quantized model: it casts layer norms to fp32 for numerical stability, enables gradient checkpointing, and marks the input embeddings so gradients can flow back through the frozen 4-bit stack to reach your adapters.

In [ ]:
# Cell C5 — Attach LoRA adapters
lora_config = LoraConfig(
    r=16,                          # rank, from cell C3
    lora_alpha=32,                 # scaling = alpha/r = 2
    target_modules="all-linear",   # every Linear layer, including q/k/v/o_proj
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model_peft = get_peft_model(model_nf4, lora_config)
model_peft.print_trainable_parameters()

Read that line carefully: **around 1.2%** of the model is trainable. About 18 million parameters out of 1.5 billion.

That is lower than the 2.08% we computed in C1 for a single square projection, and the reason is the MLP. Those matrices are rectangular — 1536 in, 8960 out — and for a rectangular layer LoRA costs `r × (d_in + d_out)` against a full `d_in × d_out`. The wider the layer, the better that ratio gets. Attention projections come in at about 2%, the MLP layers at about 1.2%, and since the MLP holds most of the weights it pulls the average down.

`target_modules="all-linear"` includes `q_proj`, `k_proj`, `v_proj` and `o_proj` — the four projections you printed in Lab 3 and found unequal, with `k_proj` seven times narrower thanks to GQA. Adapters are going onto all of them.

Let's confirm that rather than assume it.

In [ ]:
# Cell C6 — Find a real adapter inside the model
target = model_peft.base_model.model.model.layers[0].self_attn.q_proj

print(type(target).__name__)
print(f"  base weight : {tuple(target.base_layer.weight.shape)}  frozen, 4-bit")
print(f"  lora_A      : {tuple(target.lora_A['default'].weight.shape)}  trainable")
print(f"  lora_B      : {tuple(target.lora_B['default'].weight.shape)}  trainable")
print(f"\n  lora_B all zeros at init? {bool((target.lora_B['default'].weight == 0).all())}")

The same structure you built by hand in C1, now wrapped around a real quantized layer — and `lora_B` really is all zeros, exactly as the theory said.

## The training data

Ten examples. That is not a typo, and it is not enough to make this model good at anything.

What ten examples *is* enough for is proving the pipeline end to end — quantize, attach, train, save, reload, compare — in the time we have. Treat the result as a mechanism you have verified, not a model you would ship.

In [ ]:
# Cell C7 — Ten question/answer pairs about this course
training_data = [
    {"instruction": "What is quantization?",
     "response": "Quantization reduces weight precision (FP32 to INT4), shrinking memory 4-8x with minimal quality loss. NF4 is optimal for LLMs."},
    {"instruction": "What is LoRA?",
     "response": "LoRA freezes base weights and adds small trainable rank-decomposition matrices BA. Only a fraction of a percent of parameters are trained."},
    {"instruction": "What is RAG?",
     "response": "RAG retrieves relevant documents at inference time and injects them into the prompt, grounding the model in external knowledge."},
    {"instruction": "What is vLLM?",
     "response": "vLLM is a serving engine using PagedAttention for efficient KV-cache management, enabling much higher throughput than naive serving."},
    {"instruction": "What is the difference between fine-tuning and RAG?",
     "response": "Fine-tuning bakes behavior into weights, which suits style and format. RAG retrieves at runtime, which suits factual accuracy and updatable knowledge."},
    {"instruction": "What is PagedAttention?",
     "response": "PagedAttention stores the KV cache in non-contiguous memory pages, eliminating fragmentation and enabling efficient multi-user serving."},
    {"instruction": "What is QLoRA?",
     "response": "QLoRA combines a 4-bit NF4 base model with 16-bit LoRA adapters, enabling fine-tuning of large models on a single consumer GPU."},
    {"instruction": "What is a context window?",
     "response": "The maximum number of tokens a model processes in one pass, input and output combined. Overflowing it causes truncation or errors."},
    {"instruction": "What is chunking in RAG?",
     "response": "Chunking splits documents into segments before embedding. Smaller chunks improve precision; larger chunks preserve context."},
    {"instruction": "What is an embedding?",
     "response": "A dense vector encoding semantic meaning. Similar texts cluster nearby in embedding space, enabling similarity search."},
]

print(f"{len(training_data)} examples")

`SFTTrainer` wants a single `text` column holding each example as one finished string. Which means formatting them with the chat template — the same `apply_chat_template` from Lab 3, with one difference worth catching.

In [ ]:
# Cell C8 — Format each pair into one training string
def format_example(example):
    return {"text": tokenizer.apply_chat_template(
        [{"role": "user", "content": example["instruction"]},
         {"role": "assistant", "content": example["response"]}],
        tokenize=False,
        add_generation_prompt=False,      # the answer is already here; do not invite a new one
    )}


print(format_example(training_data[0])["text"])

`add_generation_prompt=False` is the difference. In Lab 3 you passed `True` at inference time, which appends a bare `<|im_start|>assistant` to tell the model it is its turn. Here the assistant's turn is already written — it is the thing we are training on — so appending another opener would teach the model to start a second empty reply.

The training string ends cleanly with `<|im_end|>`. That token is how the model learns when to stop talking.

In [ ]:
# Cell C9 — Build the dataset
from datasets import Dataset

dataset = Dataset.from_list(training_data).map(
    format_example, remove_columns=["instruction", "response"]
)

print(dataset)

In [ ]:
# Cell C10 — Training configuration
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="./lora_output",
    num_train_epochs=3,                 # 10 examples is tiny, so more than one pass
    per_device_train_batch_size=1,      # a T4 cannot hold much
    gradient_accumulation_steps=4,      # accumulate to an effective batch of 4
    learning_rate=2e-4,                 # high by full fine-tuning standards, normal for LoRA
    bf16=True,
    fp16=False,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    max_length=512,
    dataset_text_field="text",
)

print("configured")

`learning_rate=2e-4` would be reckless for full fine-tuning, where 2e-5 is more typical. It is fine here because you are not nudging pretrained weights that took months to settle — you are training two small matrices from scratch, one of which starts at zero.

`gradient_accumulation_steps=4` is the standard way around a small GPU: run four batches of one, sum the gradients, then take a single optimizer step. Same maths as a batch of four, a quarter of the memory.

In [ ]:
# Cell C11 — Train
trainer = SFTTrainer(
    model=model_peft,
    train_dataset=dataset,
    args=training_args,
    processing_class=tokenizer,
)

print("training (about 3-5 minutes on a T4)...")
trainer.train()
print("done")

Watch the loss in the log lines. With ten examples over three epochs it should fall, and it will fall a long way, because memorizing ten strings is easy. A falling loss here proves gradients are flowing into your adapters. It says nothing about whether the model learned anything general, and you should not pretend otherwise.

#### ✅ Checkpoint C

- [ ] Why is a LoRA adapter's contribution exactly zero at initialization?
- [ ] Why can the frozen base be 4-bit while the adapters stay in bf16?
- [ ] Why did we switch off the KV cache that Lab 3 argued was essential?
- [ ] What does rank control, and what would r=64 cost you?

<details>
<summary>Answers</summary>

**`lora_B` is initialized to zeros**, so `x·A·B` is zero and the model starts identical to the base. Training then departs from a known-good point.

**Because frozen weights never receive gradients.** Precision matters most where optimization happens, and that is entirely inside the adapters. The base is read-only.

**Because the cache serves generation, not training.** Training is one forward pass over a sequence already known in full — nothing to reuse. It would only waste VRAM and interfere with gradient checkpointing.

**Capacity.** r=64 is 4× the parameters, a 4× bigger adapter file, and slower training, in exchange for room to learn more complicated changes than a style shift.

</details>

---

# Part D — Ship the artifact

**~20 minutes**

Training finished. Now the question that separates a notebook from a deployment: **what exactly do you hand to someone else?**

Three answers, and you will produce all three.

In [ ]:
# Cell D1 — Save the adapter
adapter_path = "./my_lora_adapter"

model_peft.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f"saved to {adapter_path}")

In [ ]:
# Cell D2 — How big is the thing you ship?
import os

total = 0
for filename in sorted(os.listdir(adapter_path)):
    size = os.path.getsize(os.path.join(adapter_path, filename))
    total += size
    print(f"  {filename:<32s} {size/1e6:>8.2f} MB")

fp16_mb = 3100                                  # Qwen2.5-1.5B in FP16, from Part B
print(f"\nadapter package : {total/1e6:>7.1f} MB")
print(f"base in 4-bit   : {vram_nf4*1000:>7.0f} MB")
print(f"base in FP16    : {fp16_mb:>7d} MB")
print(f"\nthe adapter is ~{fp16_mb/(total/1e6):.0f}x smaller than the model it modifies")

That ratio is the entire commercial argument for adapters.

Suppose you serve forty customers, each wanting the model to behave slightly differently. Forty full fine-tunes is forty multi-gigabyte model copies to store, load, and keep in VRAM. Forty adapters is one base model plus forty small files you can swap per request.

It also makes fine-tuning behave like normal software. An adapter is small enough to put in git-lfs, attach to a pull request, version, diff by metrics, roll back, and A/B test. A 3 GB model directory is not.

Your adapter needs to say what it attaches to, though. Load it against the wrong base and you get silent nonsense, because the shapes happen to line up.

In [ ]:
# Cell D3 — A model card so the adapter is self-describing
model_card = f"""---
base_model: {MODEL_ID}
library_name: peft
tags: [qlora, fine-tuned, llm-deployment]
---

# QLoRA adapter for LLM deployment Q&A

Base model: `{MODEL_ID}`
Adapter size: {total/1e6:.1f} MB
Method: QLoRA (NF4 base + LoRA r=16, alpha=32, all-linear)
Training: 10 examples, 3 epochs, lr 2e-4

Teaching artifact from Lab 4. Ten examples is enough to verify a pipeline,
not to produce a model anyone should rely on.
"""

with open(f"{adapter_path}/README.md", "w") as f:
    f.write(model_card)

print(model_card)

The YAML block at the top is not decoration. Hugging Face Hub and most model registries parse it, and `base_model` is what lets tooling check you are attaching the adapter to the right thing. Six lines now saves a confusing afternoon later.

## Does it actually behave differently?

Time to check. Load a clean 4-bit base, ask it three questions, then attach the adapter to that same object and ask again.

In [ ]:
# Cell D4 — A fresh base model, with no adapter attached
from peft import PeftModel

del model_peft, model_nf4
gc.collect()
torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
base.config.use_cache = True          # back on: we are generating again

print("clean base model loaded")

In [ ]:
# Cell D5 — One helper for both models
def ask(model, question, max_new_tokens=150):
    messages = [{"role": "user", "content": question}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


print("ask() defined.")

In [ ]:
# Cell D6 — Record the base model's answers first
questions = ["What is QLoRA?",
             "How does LoRA reduce trainable parameters?",
             "What is NF4 quantization?"]

base_answers = {q: ask(base, q) for q in questions}

print(base_answers[questions[0]])

In [ ]:
# Cell D7 — Attach the adapter to that same base object
tuned = PeftModel.from_pretrained(base, adapter_path)

print(f"adapter attached — {total/1e6:.1f} MB added to a model already in VRAM")
print(f"VRAM now: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")

Check the VRAM number against what the base alone was using. `PeftModel.from_pretrained` did not load a second copy of 1.5 billion parameters — it hung small matrices off the layers of the model that was already there.

This is the serving pattern. One base resident in GPU memory, adapters attached and detached around it.

In [ ]:
# Cell D8 — Before and after, same questions
for question in questions:
    print(f"Q: {question}\n")
    print(f"BASE : {base_answers[question]}\n")
    print(f"TUNED: {ask(tuned, question)}")
    print("-" * 70)

Look for **style** before you look for facts. The tuned answers should come out shorter, flatter and more definitional, because all ten training examples were one-or-two-sentence definitions. That house style is what ten examples can teach in three epochs.

What it cannot teach is correctness. If a tuned answer is confidently wrong, that is the expected result of memorizing ten strings, and it is a useful thing to have seen: fine-tuning shapes *how* a model answers far more readily than *what* it knows. When you need the model to be right about facts that change, that is a retrieval problem, and it is Lab 6.

## The third artifact: a merged model

Adapters are the right default. But some runtimes want one self-contained model directory and have never heard of PEFT — llama.cpp and Ollama among them. For those you merge the adapter into the weights.

One subtlety: merge into an **FP16** base, not the 4-bit one. Adding bf16 adapter weights into 4-bit quantized weights and re-quantizing compounds the error. The NF4 model was the right tool for training cheaply; FP16 is the right base for producing a clean export.

In [ ]:
# Cell D9 — Merge into an FP16 base for export
del base, tuned
gc.collect()
torch.cuda.empty_cache()

merge_base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")
merged = PeftModel.from_pretrained(merge_base, adapter_path).merge_and_unload()

merged_path = "./my_merged_model"
merged.save_pretrained(merged_path)
tokenizer.save_pretrained(merged_path)

size = sum(os.path.getsize(os.path.join(merged_path, f)) for f in os.listdir(merged_path))
print(f"merged model: {size/1e9:.2f} GB at {merged_path}")
print(f"adapter     : {total/1e6:.1f} MB")

`merge_and_unload()` computes `W + (alpha/r)·B·A` for every adapted layer, writes the result back into the weight, and removes the LoRA wrappers. What comes out is an ordinary model that happens to have your fine-tuning baked in. Nothing downstream needs to know PEFT exists.

You have now produced all three artifacts:

| Artifact | Size | Use it when |
|---|---|---|
| `./my_lora_adapter` | ~85 MB | Serving many variants from one base. The default. |
| `./my_merged_model` | ~3 GB | The runtime wants a plain model directory, or you are converting to another format |
| GGUF (below) | ~1 GB at Q4_K_M | Running on a laptop CPU through llama.cpp, Ollama or LM Studio |

### The GGUF path

Converting needs llama.cpp built, which takes longer than we have. Keep this as a recipe — Bonus Lab 06 walks through Ollama properly.

```bash
git clone https://github.com/ggml-org/llama.cpp && cd llama.cpp
uv pip install -r requirements.txt
python convert_hf_to_gguf.py ../my_merged_model --outfile ../my-qlora-f16.gguf --outtype f16

cmake -B build && cmake --build build --config Release -j
./build/bin/llama-quantize ../my-qlora-f16.gguf ../my-qlora-q4_k_m.gguf Q4_K_M
```

Then a three-line `Modelfile`:

```
FROM ./my-qlora-q4_k_m.gguf
PARAMETER temperature 0.2
SYSTEM You are an assistant fine-tuned on LLM deployment concepts.
```

```bash
ollama create llm-deploy-qlora -f Modelfile && ollama run llm-deploy-qlora
```

Note what `Q4_K_M` is doing at the end there: quantizing again, to 4 bits, using a k-quant scheme that is a cousin of the NF4 work you did in Part A. Same idea, different runtime.

#### ✅ Checkpoint D

- [ ] How many times smaller is the adapter than its base?
- [ ] Why does `PeftModel.from_pretrained` barely move the VRAM number?
- [ ] Why merge into FP16 rather than the NF4 model you trained against?
- [ ] Which of the three artifacts would you ship to serve 40 customers?

<details>
<summary>Answers</summary>

**Tens of times.** With `r=16` on `all-linear` the adapter lands around 75 MB against a 3.1 GB FP16 base, so roughly 35-40x. Drop to `r=4`, or adapt only `q_proj` and `v_proj`, and it shrinks by a lot more.

**Because it attaches to the model already in memory.** Only the small A and B matrices are loaded; the 1.5B base is reused in place.

**Merging bf16 adapters into 4-bit weights and re-quantizing stacks error on error.** FP16 gives a clean merge.

**The adapter.** One base resident in VRAM, forty small files swapped per request, each versioned and rolled back like ordinary code.

</details>

---

# Bonus — Pruning, and why it loses

**~8 minutes**

Quantization makes every weight cheaper. **Pruning** takes the opposite approach: delete the weights that appear not to matter, keep the rest at full precision.

The intuition is appealing. Most weights are near zero — you measured that in Part A — so surely zeroing the smallest ones costs little? Let's find out, and then look at what actually happens to the file on disk.

In [ ]:
# Cell E1 — A fresh small model to vandalize
del merged, merge_base
gc.collect()
torch.cuda.empty_cache()

demo = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", dtype=torch.bfloat16)
demo_tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

def sparsity(model):
    total = zeros = 0
    for p in model.parameters():
        total += p.numel()
        zeros += (p == 0).sum().item()
    return zeros / total

print(f"sparsity before pruning: {sparsity(demo):.2%}")

In [ ]:
# Cell E2 — Zero the smallest 30% of all Linear weights
import torch.nn.utils.prune as prune_utils

linear_layers = [(m, "weight") for m in demo.modules() if isinstance(m, torch.nn.Linear)]

prune_utils.global_unstructured(linear_layers,
                                pruning_method=prune_utils.L1Unstructured,
                                amount=0.3)
for module, name in linear_layers:
    prune_utils.remove(module, name)      # make the mask permanent

print(f"sparsity after pruning : {sparsity(demo):.2%}")
print(f"layers pruned          : {len(linear_layers)}")

Thirty percent of the weights in every linear layer are now exactly zero. Global magnitude pruning ranks all of them together and cuts the smallest, so layers that were tolerant give up more than layers that were not.

Does it still work?

In [ ]:
# Cell E3 — Quality after pruning
def quick_answer(model, tok, prompt, max_new_tokens=40):
    formatted = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                        tokenize=False, add_generation_prompt=True)
    inputs = tok(formatted, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


print(quick_answer(demo, demo_tok, "What is quantization? One sentence."))

In [ ]:
# Cell E4 — And what did it save?
pruned_path = "./pruned_demo"
demo.save_pretrained(pruned_path)

size = sum(os.path.getsize(os.path.join(pruned_path, f)) for f in os.listdir(pruned_path))
print(f"sparsity on disk : {sparsity(demo):.1%} of weights are zero")
print(f"file size        : {size/1e9:.2f} GB")
print(f"original FP16    : ~0.99 GB")

There it is. Thirty percent of the weights are zero, and **the file is the same size.**

A zero in a dense tensor is a number like any other. It occupies its two bytes, it gets loaded into VRAM, and the GPU multiplies by it just as eagerly as by any other value. Nothing about writing `0.0` into a `float16` slot makes that slot cheaper.

To convert sparsity into savings you need something that exploits it:

- **A sparse storage format** — store only the non-zeros plus their indices. The indices cost memory too, so below roughly 70% sparsity you can end up *larger*.
- **Sparse kernels** — hardware that skips the zeros. NVIDIA's 2:4 structured sparsity does this, but only in a rigid pattern (two zeros in every group of four), which is not what magnitude pruning produces.
- **Retraining afterward** to recover the quality you just lost.

Now compare that against what you did in Part B: `load_in_4bit=True`, one argument, a third of the memory, on any GPU, with quality you could not distinguish by eye.

That is the whole comparison. Pruning zeroes weights without shrinking the file and without running faster, unless you have specialised hardware and a matching sparsity pattern. Quantization shrinks the file on hardware you already own.

**For LLM deployment: quantize first. Reach for pruning rarely, and only with the kernels to back it up.**

### Where sparsity actually pays off

Not in pruning, but in **Mixture of Experts**. An MoE model replaces each feed-forward block with many "expert" blocks and a router that activates only two or so per token. Mixtral 8x7B holds 47B parameters but computes with about 13B on any given token.

That is sparsity with a payoff, because it is **structured by design** rather than discovered by thresholding. The model is expensive to hold in memory and cheap to run — the opposite trade from quantization, and increasingly the shape of frontier models.

---

## ✅ Lab 4 complete

- [ ] Quantized a real weight tensor by hand and watched one outlier ruin a whole-tensor scale
- [ ] Measured block-wise quantization as ~10× more accurate at the same bit width
- [ ] Worked out why "4-bit" is really 4.25 bits, and what double quantization recovers
- [ ] Showed NF4's quantile-spaced levels beat evenly spaced ones by ~20% error
- [ ] Measured FP16 against NF4 on a T4 for memory, speed and quality
- [ ] Explained why the smaller model was the slower one
- [ ] Sized a LoRA adapter and proved it contributes exactly zero at initialization
- [ ] Trained a 1.5B model on a GPU that could not have held its optimizer state
- [ ] Saved an adapter, a model card, and a merged export
- [ ] Compared base against tuned and described what actually changed
- [ ] Zeroed 30% of a model's weights and watched the file size not move

## What to take with you

1. **Quantization buys fit, not speed.** Sixteen levels, placed at normal quantiles, scaled per 64-weight block. The weights unpack to bf16 for every matmul, which is why NF4 can be slower than FP16 and still be the right call.
2. **Block size is the whole game.** One 18-sigma outlier destroys a shared scale. Small blocks contain the damage; the scales themselves then become worth compressing.
3. **LoRA works because ΔW is low rank.** Freeze the base, learn two thin matrices, start from zero so the model begins exactly where you trusted it.
4. **QLoRA composes because frozen weights do not need precision.** Spend your bits where the gradients are.
5. **The adapter is the deliverable.** Small enough to version, review, roll back and A/B test. Merge only when the runtime demands one directory.
6. **Fine-tuning changes how a model answers more than what it knows.** For facts, retrieve — that is Lab 6.

## Stretch goals

1. **Rank sweep.** Retrain with `r=4` and again with `r=64`. Compare adapter size, training time, and whether the tuned answers actually improve. Most tasks plateau earlier than people expect.
2. **Break the quantization.** In cell A4, try `block_size=4096`. How bad does the error get, and where does it stop mattering?
3. **Merged vs attached at inference.** Benchmark `merge_and_unload()` output against base-plus-adapter with the Part B helper. Is the fused version measurably faster?
4. **Push it.** `model_peft.push_to_hub("your-username/my-qlora-adapter")` after `hf auth login`. The model card you wrote in D3 goes with it.
5. **Double quantization off.** Reload with `bnb_4bit_use_double_quant=False` and measure the VRAM difference. Does it match the ~0.4 bits/weight estimate from Part A?

## Next

[Lab 5 — Serving API](../05_Serving_API/README.md) is Day 2, back on CPU. You will put a model behind an OpenAI-compatible FastAPI server — the `base_url` swap from Lab 1A, except the URL is now yours. Bring the ngrok authtoken.

The thread running through it: Lab 3 told you what a model costs to hold, Lab 4 made it fit and taught it something, Lab 5 puts it behind an endpoint.